# MOVIE CREATOR — All-in-One Studio sa Google Colab (T4)

Isang notebook para patakbuhin ang **buong studio** sa libreng Colab T4 GPU — walang Server-2.

**Ano ang binubuo nito**

| Bahagi | Ano |
|---|---|
| App | MOVIE CREATOR (Node 22 + Vite build) sa `127.0.0.1:3000` |
| Video engine | ComfyUI + Wan 2.1 T2V 1.3B sa `127.0.0.1:8188` |
| Public link | Isang Cloudflare quick tunnel papunta sa app |

**Paano gamitin**

1. `Runtime → Change runtime type → T4 GPU`
2. `Runtime → Run all`
3. Hintayin ang huling cell — ipi-print nito ang **`SERVER_URL`** at **`APP_AUTH_TOKEN`**
4. Ilagay ang dalawa sa **Server Setup** ng Android APK

> ⚠️ **Kailangan ng T4.** Ang free CPU runtime ay hindi kayang mag-render ng video.
> ⚠️ **Hindi tumatagal ang session.** Kapag nag-disconnect ang Colab, patay ang app at ang tunnel. I-`Run all` lang muli — bubuo ito ng bagong `SERVER_URL`.

**Disk:** ~10 GB ang weights (2.84 + 6.74 + 0.25 GB). Sapat ang ~110 GB ng Colab.


## 1. Suriin ang T4 GPU

Titigil agad ang notebook kung walang GPU, para hindi ka maghintay ng 10 GB download tapos mabibigo lang.


In [ ]:
import subprocess, sys, os

def sh(cmd, **kw):
    """Run a shell command, echo it, and raise on failure."""
    print(f"$ {cmd}")
    return subprocess.run(cmd, shell=True, check=True, **kw)

smi = subprocess.run("nvidia-smi", shell=True, capture_output=True, text=True)
if smi.returncode != 0:
    sys.exit(
        "WALANG GPU.\n"
        "Pumunta sa Runtime > Change runtime type > Hardware accelerator > T4 GPU, "
        "tapos i-Run all muli. Kailangan ng GPU para sa ComfyUI."
    )
print(smi.stdout)

gpu = subprocess.run(
    "nvidia-smi --query-gpu=name,memory.total --format=csv,noheader",
    shell=True, capture_output=True, text=True,
).stdout.strip()
print("GPU:", gpu)
if "T4" not in gpu:
    print(f"PAALALA: inaasahan ang T4, nakuha: {gpu}. Pwede pa ring tumakbo kung may sapat na VRAM.")

total_ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1e9
disk = subprocess.run("df -BG --output=avail /content | tail -1", shell=True, capture_output=True, text=True).stdout.strip()
print(f"System RAM: {total_ram:.1f} GB | Free disk sa /content: {disk}")


## 2. Node.js 22+ at ang MOVIE CREATOR repo

Ang app ay `"type": "module"` na TypeScript, at `npm start` ay direktang `node server.ts`.
Kailangan ng **Node 22+** para sa built-in type stripping. Kung mas luma ang Node, gagamit tayo ng `npm run dev` (tsx) bilang fallback.


In [ ]:
# Node 22 (Colab ships an old Node by default).
sh("curl -fsSL https://deb.nodesource.com/setup_22.x | bash -")
sh("apt-get install -y nodejs > /dev/null")
node_major = int(subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip().lstrip("v").split(".")[0])
print("node:", subprocess.run("node -v", shell=True, capture_output=True, text=True).stdout.strip())
print("npm :", subprocess.run("npm -v", shell=True, capture_output=True, text=True).stdout.strip())
if node_major < 22:
    print("PAALALA: Node < 22 — gagamitin ang `npm run dev` (tsx) imbes na `npm start`.")

REPO_DIR = "/content/MOVIES"
if not os.path.isdir(REPO_DIR):
    sh(f"git clone --branch main --depth 1 https://github.com/Rex217-0/MOVIES.git {REPO_DIR}")
else:
    sh(f"git -C {REPO_DIR} fetch --depth 1 origin main")
    sh(f"git -C {REPO_DIR} reset --hard origin/main")
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())
print(subprocess.run("git log -1 --oneline", shell=True, capture_output=True, text=True).stdout.strip())


In [ ]:
sh("npm install --no-audit --no-fund 2>&1 | tail -5")
sh("npm run build 2>&1 | tail -8")
print("BUILD OK —", subprocess.run("ls -la dist | head -3", shell=True, capture_output=True, text=True).stdout)


## 3. ComfyUI + VideoHelperSuite + cloudflared

Ang `VHS_VideoCombine` node (galing sa VideoHelperSuite) ang nagsusulat ng `.mp4` sa dulo ng workflow, kaya kailangan ito — hindi optional.


In [ ]:
COMFY_DIR = "/content/ComfyUI"
if not os.path.isdir(COMFY_DIR):
    sh(f"git clone --depth 1 https://github.com/comfyanonymous/ComfyUI.git {COMFY_DIR}")

sh(f"pip install -q -r {COMFY_DIR}/requirements.txt")
sh(f"pip install -q soundfile")

VHS_DIR = f"{COMFY_DIR}/custom_nodes/ComfyUI-VideoHelperSuite"
if not os.path.isdir(VHS_DIR):
    sh(f"git clone --depth 1 https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite.git {VHS_DIR}")
sh(f"pip install -q -r {VHS_DIR}/requirements.txt")

if not os.path.exists("/usr/local/bin/cloudflared"):
    sh("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared")
    sh("chmod +x /usr/local/bin/cloudflared")
print(subprocess.run("cloudflared --version", shell=True, capture_output=True, text=True).stdout.strip())

for sub in ("diffusion_models", "text_encoders", "vae"):
    os.makedirs(f"{COMFY_DIR}/models/{sub}", exist_ok=True)
print("ComfyUI dirs ready")


## 4. Wan 2.1 1.3B weights — direkta sa tamang folder

Ang Hugging Face repo (`Comfy-Org/Wan-2.1`) ay naglalagay ng mga file sa loob ng `split_files/`.
**Hindi** natin kokopyahin ang `split_files/` prefix — direktang ilalagay sa folder na hinahanap ng ComfyUI,
kaya tugma ang pangalan sa `COMFYUI_CHECKPOINT` at sa workflow.

Bawat file ay sinusuri ang laki pagkatapos i-download; hindi lalampas kung kulang (putol na download).


In [ ]:
WEIGHTS = [
    ("diffusion_models", "wan2.1_t2v_1.3B_fp16.safetensors", 2838303560),
    ("text_encoders",    "umt5_xxl_fp8_e4m3fn_scaled.safetensors", 6735906897),
    ("vae",              "wan_2.1_vae.safetensors", 253815318),
]
BASE = "https://huggingface.co/Comfy-Org/Wan-2.1/resolve/main/split_files"

for folder, name, expected in WEIGHTS:
    dest = f"{COMFY_DIR}/models/{folder}/{name}"
    if os.path.exists(dest) and os.path.getsize(dest) == expected:
        print(f"OK (nandiyan na) {folder}/{name}")
        continue
    print(f"\nDownloading {folder}/{name} ({expected/1e9:.2f} GB)...")
    sh(f'wget -q --show-progress -c "{BASE}/{folder}/{name}" -O "{dest}"')
    got = os.path.getsize(dest)
    if got != expected:
        sys.exit(f"PUTOL ang download: {name} — inaasahan {expected} bytes, nakuha {got}. I-run muli ang cell na ito.")
    print(f"OK {folder}/{name} — {got} bytes")

print("\nLahat ng weights kumpleto.")


## 5. Patakbuhin ang ComfyUI sa `127.0.0.1:8188`

Hihintayin natin ang `/system_stats` na mag-200 — iyon ang eksaktong probe na ginagamit ng app
(`server.ts`), kaya kung pumasa ito, makikita ng app ang ComfyUI.


In [ ]:
COMFY_LOG = "/content/comfyui.log"
comfy_proc = subprocess.Popen(
    f"cd {COMFY_DIR} && python main.py --listen 127.0.0.1 --port 8188 > {COMFY_LOG} 2>&1",
    shell=True,
)

import json, time, urllib.request, urllib.error

def wait_http(url, seconds=600, label=""):
    """Poll a URL until it returns HTTP 200."""
    deadline = time.time() + seconds
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=5) as r:
                if r.status == 200:
                    print(f"{label or url} -> 200")
                    return True
        except (urllib.error.URLError, urllib.error.HTTPError, OSError):
            pass
        time.sleep(5)
    return False

if not wait_http("http://127.0.0.1:8188/system_stats", 600, "ComfyUI /system_stats"):
    print(subprocess.run(f"tail -40 {COMFY_LOG}", shell=True, capture_output=True, text=True).stdout)
    sys.exit("Bumagsak ang ComfyUI. Tingnan ang log sa itaas.")

stats = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/system_stats", timeout=10).read())
dev = stats["devices"][0]
print(f"ComfyUI OK — {dev['name']} | VRAM {dev['vram_total']/1e9:.1f} GB")

# The workflow needs these exact node/model names; prove they are visible before starting the app.
info = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/object_info", timeout=30).read())
needed_nodes = ["UNETLoader", "CLIPLoader", "VAELoader", "CLIPTextEncode",
                "EmptyHunyuanLatentVideo", "KSampler", "VAEDecode", "VHS_VideoCombine"]
missing = [n for n in needed_nodes if n not in info]
if missing:
    sys.exit(f"KULANG ang nodes sa ComfyUI: {missing}. Kadalasan ito ay VideoHelperSuite na hindi na-install.")

ckpts = info["UNETLoader"]["input"]["required"]["unet_name"][0]
print("UNET names ComfyUI can see:", ckpts)
if "wan2.1_t2v_1.3B_fp16.safetensors" not in ckpts:
    sys.exit("Hindi makita ng ComfyUI ang Wan 2.1 UNET. Tiyakin nasa models/diffusion_models/ ito.")
print("Lahat ng node at model na kailangan ay nakita.")


## 6. Buuin ang `.env`

Random ang `APP_AUTH_TOKEN` — hindi hardcoded, at hindi ipi-print maliban sa huling cell.
Kung may `GEMINI_API_KEY` sa Colab Secrets, gagamitin; kung wala, tuloy pa rin ang app (free path).


In [ ]:
import secrets

PORT = 3000
APP_AUTH_TOKEN = secrets.token_urlsafe(32)

try:
    from google.colab import userdata
    try:
        gemini_key = userdata.get("GEMINI_API_KEY")
    except Exception:
        gemini_key = None
except ImportError:
    gemini_key = None

env_lines = [
    f"PORT={PORT}",
    f"APP_AUTH_TOKEN={APP_AUTH_TOKEN}",
    "PREFERRED_RENDER_ENGINE=comfyui",
    "COMFYUI_URL=http://127.0.0.1:8188",
    "COMFYUI_CHECKPOINT=wan2.1_t2v_1.3B_fp16.safetensors",
    "COMFYUI_VIDEO_WORKFLOW=./workflows/comfyui_video_api.json",
    "ALLOW_PAID_CLOUD=0",
]
if gemini_key:
    env_lines.append(f"GEMINI_API_KEY={gemini_key}")
    print("GEMINI_API_KEY: nakuha mula sa Colab Secrets")
else:
    print("GEMINI_API_KEY: wala — free path lang (ok lang ito)")

env_text = "\n".join(env_lines) + "\n"
with open(f"{REPO_DIR}/.env", "w") as f:
    f.write(env_text)
os.chmod(f"{REPO_DIR}/.env", 0o600)

print("\n.env nakasulat (nakatago ang token):")
for line in env_lines:
    k = line.split("=", 1)[0]
    print(f"  {k}=" + ("<hidden>" if k in ("APP_AUTH_TOKEN", "GEMINI_API_KEY") else line.split('=', 1)[1]))


## 7. Patakbuhin ang app at buksan ang Cloudflare tunnel

**Isang** tunnel lang papunta sa `127.0.0.1:3000`. Ang URL na `trycloudflare.com` ang siyang ilalagay sa APK.


In [ ]:
APP_LOG = "/content/app.log"

# Production mode serves the built dist/ and refuses to start without an
# APP_AUTH_TOKEN. Both hold here: Section 2 ran `npm run build` and Section 6
# wrote the token. The dev fallback keeps NODE_ENV unset because it needs the
# Vite middleware instead of dist/.
if node_major >= 22:
    start_cmd = "npm start"
    app_env = "NODE_ENV=production "
    if not os.path.isdir(f"{REPO_DIR}/dist"):
        sys.exit("Walang dist/ — patakbuhin muna ang Section 2 (`npm run build`).")
else:
    start_cmd = "npm run dev"
    app_env = ""

app_proc = subprocess.Popen(
    f"cd {REPO_DIR} && {app_env}{start_cmd} > {APP_LOG} 2>&1",
    shell=True,
)

if not wait_http(f"http://127.0.0.1:{PORT}/", 300, "MOVIE CREATOR /"):
    print(subprocess.run(f"tail -40 {APP_LOG}", shell=True, capture_output=True, text=True).stdout)
    sys.exit(f"Bumagsak ang app. Tingnan ang {APP_LOG}.")
print(f"App tumatakbo gamit ang `{app_env}{start_cmd}`")


In [ ]:
TUNNEL_LOG = "/content/cloudflared.log"
tunnel_proc = subprocess.Popen(
    f"cloudflared tunnel --url http://127.0.0.1:{PORT} --no-autoupdate > {TUNNEL_LOG} 2>&1",
    shell=True,
)

import re
SERVER_URL = None
deadline = time.time() + 180
while time.time() < deadline and SERVER_URL is None:
    if os.path.exists(TUNNEL_LOG):
        found = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open(TUNNEL_LOG).read())
        if found:
            SERVER_URL = found[-1]
            break
    time.sleep(3)

if not SERVER_URL:
    print(open(TUNNEL_LOG).read()[-2000:])
    sys.exit("Walang nakuha na trycloudflare.com URL. Tingnan ang log sa itaas.")

print("Tunnel:", SERVER_URL)

# The tunnel is only useful if it actually reaches the app through the public edge.
if not wait_http(f"{SERVER_URL}/android", 180, "Public URL /android"):
    sys.exit("Bukas ang tunnel pero hindi maabot ang app sa public URL. I-run muli ang cell na ito.")


## 8. Ito ang kailangan mo — ilagay sa Server Setup ng APK

Ang `APP_AUTH_TOKEN` ay ang **tanging** kopya na makikita mo. Kung mawala, patakbuhin muli ang Section 6.


In [ ]:
print("=" * 62)
print("MOVIE CREATOR — READY")
print("=" * 62)
print()
print("SERVER_URL:")
print(SERVER_URL)
print()
print("APP_AUTH_TOKEN:")
print(APP_AUTH_TOKEN)
print()
print("=" * 62)
print("Buksan ang app sa phone -> Server Setup -> ilagay ang dalawa sa itaas.")
print("Kapag nag-disconnect ang Colab, patay ang URL: i-Run all muli.")
print("=" * 62)


## 9. Sariling pagsusuri — huwag masira ang repo

Ito ang parehong tatlong check na tumatakbo sa CI. Dapat pumasa bago mo itulak ang anumang pagbabago.


In [ ]:
checks = {}
for label, cmd in (("lint", "npm run lint"), ("build", "npm run build"), ("test", "npm test")):
    print(f"\n===== {label} =====")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    checks[label] = r.returncode
    tail = (r.stdout + r.stderr).strip().split("\n")
    print("\n".join(tail[-12:]))
    print(f"-> {label} exit {r.returncode}")

print("\n===== BUOD =====")
for k, v in checks.items():
    print(f"  {k}: {'PASS' if v == 0 else 'FAIL'} (exit {v})")
if any(checks.values()):
    sys.exit("May bumagsak na check. Huwag mag-commit hangga't hindi pumapasa.")
print("Lahat pumasa.")


## 10. (Opsyonal) I-commit ang notebook na ito

Kailangan ng `GITHUB_TOKEN` sa Colab Secrets na may `repo` scope. Kung wala, laktawan lang — hindi ito kailangan para tumakbo ang studio.


In [ ]:
NOTEBOOK_PATH = f"{REPO_DIR}/scripts/colab_all_in_one_studio.ipynb"

try:
    from google.colab import userdata
    gh_token = userdata.get("GITHUB_TOKEN")
except Exception:
    gh_token = None

if not gh_token:
    print("Walang GITHUB_TOKEN sa Colab Secrets — nilalaktawan ang commit.")
    print(f"Ang notebook ay nasa {NOTEBOOK_PATH} kung kailangan mo itong i-download.")
else:
    sh(f'git -C {REPO_DIR} config user.name "OpenHands"')
    sh(f'git -C {REPO_DIR} config user.email "openhands@all-hands.dev"')
    sh(f"git -C {REPO_DIR} add scripts/colab_all_in_one_studio.ipynb")
    status = subprocess.run(f"git -C {REPO_DIR} status --porcelain", shell=True, capture_output=True, text=True).stdout
    if not status.strip():
        print("Walang pagbabago — commit na ito dati pa.")
    else:
        sh(f'git -C {REPO_DIR} commit -m "docs(colab): all-in-one studio notebook (T4 GPU, no Server-2)"')
        sh(f"git -C {REPO_DIR} push https://{gh_token}@github.com/Rex217-0/MOVIES.git main")
        print("Na-push sa main.")


## Troubleshooting

| Sintomas | Gawin |
|---|---|
| `WALANG GPU` | Runtime → Change runtime type → T4 GPU, tapos Run all |
| ComfyUI timeout | `!tail -40 /content/comfyui.log` — madalas OOM o kulang na disk |
| `KULANG ang nodes` | `pip install -r /content/ComfyUI/custom_nodes/ComfyUI-VideoHelperSuite/requirements.txt` tapos i-restart ang ComfyUI |
| `Hindi makita ... UNET` | Nasa `ComfyUI/models/diffusion_models/` ba? Walang `split_files/` prefix |
| `PUTOL ang download` | I-run muli ang Section 4 — `wget -c` ang nagpapatuloy |
| Hindi bumukas ang tunnel | Patay na ang app; tingnan `/content/app.log` |
| Nag-disconnect ang Colab | Patay ang URL. I-Run all muli — **bagong** URL at **bagong** token |

### Hindi pa napatunayan

Ang Wan 2.1 graph ay **hindi pa** naipatakbo sa totoong ComfyUI mula sa dev machine.
Section 5 ay nagpapatunay na nakikita ang lahat ng node at model, pero ang totoong render
ay sa unang job mo pa lang makikita. Kung may error ang graph, ang `/content/comfyui.log` ang magsasabi.
